# 📊 Relatório Consolidado — Comparação de Modelos
**Dataset:** Sonar – Classificação de Minas vs Rochas  
**Modelos avaliados:** Árvore de Decisão | KNN (k=5) | Regressão Logística  


## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120


## 2. Tabela Consolidada de Métricas

> **Instruções:** substitua os valores abaixo pelos gerados em cada notebook individual.


In [ ]:
# ── Substitua pelos valores reais dos seus notebooks ──────────────────────────
dados = {
    "Modelo"          : ["Árvore de Decisão", "KNN (k=5)", "Regressão Logística"],
    "Acurácia"        : [0.0000, 0.0000, 0.0000],   # ← preencher
    "Precisão"        : [0.0000, 0.0000, 0.0000],
    "Recall"          : [0.0000, 0.0000, 0.0000],
    "F1-Score"        : [0.0000, 0.0000, 0.0000],
    "ROC-AUC"         : [0.0000, 0.0000, 0.0000],
    "CV Acurácia (μ)" : [0.0000, 0.0000, 0.0000],
}
# ──────────────────────────────────────────────────────────────────────────────

df_comp = pd.DataFrame(dados)
df_comp = df_comp.set_index("Modelo")

# Destaque do melhor em cada métrica
styled = df_comp.style.highlight_max(axis=0, color="#d4edda")                       .highlight_min(axis=0, color="#f8d7da")                       .format("{:.4f}")

print("=== TABELA CONSOLIDADA DE MÉTRICAS ===")
display(styled)


## 3. Gráfico Comparativo de Métricas

In [ ]:
metricas_plot = ["Acurácia", "Precisão", "Recall", "F1-Score", "ROC-AUC"]
modelos  = df_comp.index.tolist()
cores    = ["#2196F3", "#4CAF50", "#9C27B0"]

x   = np.arange(len(metricas_plot))
w   = 0.25
fig, ax = plt.subplots(figsize=(12, 5))

for i, (modelo, cor) in enumerate(zip(modelos, cores)):
    vals = [df_comp.loc[modelo, m] for m in metricas_plot]
    bars = ax.bar(x + i*w - w, vals, w, label=modelo, color=cor, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metricas_plot, fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Comparação de Métricas por Modelo – Dataset Sonar', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('comparativo_metricas.png', bbox_inches='tight')
plt.show()


## 4. Radar Chart (Visão Holística)

In [ ]:
from matplotlib.patches import FancyArrowPatch

cats   = ["Acurácia", "Precisão", "Recall", "F1-Score", "ROC-AUC"]
N      = len(cats)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
cores = ["#2196F3", "#4CAF50", "#9C27B0"]

for modelo, cor in zip(modelos, cores):
    vals = [df_comp.loc[modelo, c] for c in cats]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', lw=2, color=cor, label=modelo)
    ax.fill(angles, vals, alpha=0.08, color=cor)

ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, fontsize=11)
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_title('Radar – Comparação Holística', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('radar_comparativo.png', bbox_inches='tight')
plt.show()


## 5. Análise de Contexto: Recall e FN

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║          ANÁLISE DE CONTEXTO — DETECÇÃO DE MINAS           ║
╠══════════════════════════════════════════════════════════════╣
║  Neste problema, um FALSO NEGATIVO (mina classificada como ║
║  rocha) é crítico — pode resultar em risco de vida.        ║
║                                                            ║
║  Portanto, além da Acurácia, o RECALL (Sensibilidade)      ║
║  é a métrica mais relevante para o contexto operacional.   ║
║                                                            ║
║  Um modelo com Recall alto sacrifica menos minas reais.    ║
╚══════════════════════════════════════════════════════════════╝
""")

melhor_recall = df_comp["Recall"].idxmax()
melhor_f1     = df_comp["F1-Score"].idxmax()
melhor_acc    = df_comp["Acurácia"].idxmax()

print(f"🏆 Melhor Recall   : {melhor_recall}  ({df_comp.loc[melhor_recall, 'Recall']:.4f})")
print(f"🏆 Melhor F1-Score : {melhor_f1}     ({df_comp.loc[melhor_f1, 'F1-Score']:.4f})")
print(f"🏆 Melhor Acurácia : {melhor_acc}    ({df_comp.loc[melhor_acc, 'Acurácia']:.4f})")


## 6. Ranking Final

In [ ]:
# Score ponderado: damos mais peso ao Recall (contexto de segurança)
df_rank = df_comp.copy()
df_rank["Score Ponderado"] = (
    df_rank["Acurácia"]  * 0.15 +
    df_rank["Precisão"]  * 0.15 +
    df_rank["Recall"]    * 0.30 +   # peso maior
    df_rank["F1-Score"]  * 0.25 +
    df_rank["ROC-AUC"]   * 0.15
)

df_rank = df_rank.sort_values("Score Ponderado", ascending=False)
df_rank["Ranking"] = range(1, len(df_rank)+1)

print("=== RANKING FINAL (ponderado pelo contexto) ===")
display(df_rank[["Ranking","Acurácia","Recall","F1-Score","Score Ponderado"]].style.format("{:.4f}"))

print(f"\n✅ Modelo recomendado: {df_rank.index[0]}")
print(f"   Justificativa: melhor equilíbrio entre Recall e F1-Score no contexto de detecção de minas.")


## 7. Conclusão

| Critério de Aceite | Status |
|---|---|
| Métricas geradas para todos os modelos | ✅ Acurácia, Precisão, Recall, F1, ROC-AUC, CV |
| Comparação consolidada | ✅ Tabela + Gráfico de barras + Radar |
| Resultados documentados | ✅ Análise contextual + Ranking ponderado |
| Validação cruzada | ✅ 5-fold em todos os modelos |
| Análise de contexto (FN crítico) | ✅ Seção 5 |
